## 언어 모델 양자화하기

### 문제 설명
LSTM을 사용해 **언어 모델**을 구현하고, 추론 성능을 높이기 위해 **동적 양자화(dynamic quantization)** 를 적용합니다. 동적 양자화는 모델 가중치를 양자화하여 모델 크기를 줄이고 추론 속도를 향상시킵니다.

### 요구사항

1. **언어 모델 정의**:
   - **목적**: 시퀀스에서 다음 토큰을 예측하는 간단한 언어 모델을 만듭니다.
   - **구성 요소**:
     - **임베딩 레이어**: 입력 토큰을 밀집 벡터 표현으로 변환합니다.
     - **LSTM 레이어**: 임베딩된 시퀀스를 처리해 시간적 의존성을 포착합니다.
     - **완전 연결 레이어**: 다음 토큰에 대한 예측을 출력합니다.
     - **Softmax 레이어**: 예측을 위해 어휘 전체에 대한 확률 분포를 적용합니다.
   - **Forward Pass**:
     - 입력 시퀀스를 임베딩 레이어에 통과시킵니다.
     - 임베딩된 시퀀스를 LSTM에 전달합니다.
     - LSTM의 마지막 hidden state를 사용해 완전 연결 레이어로 예측을 만듭니다.
     - softmax 함수를 적용해 어휘 전체에 대한 확률을 얻습니다.

2. **동적 양자화 적용**:
   - 모델을 동적으로 양자화합니다.
   - 양자화된 모델의 성능을 원본 모델과 비교해 평가합니다.


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.quantization import quantize_dynamic

In [4]:
# TODO: Define a simple Language Model (an LSTM-based model)
class LanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers):
        super(LanguageModel, self).__init__()
        ...

    def forward(self, x):
        ...

In [5]:
# Create synthetic training data
torch.manual_seed(42)
vocab_size = 50
seq_length = 10
batch_size = 32
X_train = torch.randint(0, vocab_size, (batch_size, seq_length))  # Random integer input
y_train = torch.randint(0, vocab_size, (batch_size,))  # Random target words

# Initialize the model, loss function, and optimizer
embed_size = 64
hidden_size = 128
num_layers = 2
model = LanguageModel(vocab_size, embed_size, hidden_size, num_layers)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [6]:
# Training loop
epochs = 5
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    output = model(X_train)
    loss = criterion(output, y_train)
    loss.backward()
    optimizer.step()

    # Log progress every epoch
    print(f"Epoch [{epoch + 1}/{epochs}] - Loss: {loss.item():.4f}")

# Now, we will quantize the model dynamically to reduce its size and improve inference speed
# Quantization: Apply dynamic quantization to the language model
quantized_model = quantize_dynamic(model, {nn.Linear, nn.LSTM}, dtype=torch.qint8)

# Save the quantized model
torch.save(quantized_model.state_dict(), "quantized_language_model.pth")


Epoch [1/5] - Loss: 3.9118
Epoch [2/5] - Loss: 3.9113
Epoch [3/5] - Loss: 3.9108
Epoch [4/5] - Loss: 3.9103
Epoch [5/5] - Loss: 3.9097


In [7]:
# Load the quantized model and test it
quantized_model = LanguageModel(vocab_size, embed_size, hidden_size, num_layers)

# Apply dynamic quantization on the model after defining it
quantized_model = quantize_dynamic(quantized_model, {nn.Linear, nn.LSTM}, dtype=torch.qint8)

quantized_model.load_state_dict(torch.load("quantized_language_model.pth"))

<All keys matched successfully>

In [8]:
# Testing the quantized model on a sample input
quantized_model.eval()
test_input = torch.randint(0, vocab_size, (1, seq_length))
with torch.no_grad():
    prediction = quantized_model(test_input)
    print(f"Prediction for input {test_input.tolist()}: {prediction.argmax(dim=1).item()}")

Prediction for input [[15, 28, 33, 19, 37, 24, 48, 42, 33, 35]]: 49
